In [124]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Load your dataset
df = pd.read_csv('fraud_dataset.csv')

# 2. Encode Categorical Columns
cat_cols = ['Merchant', 'Location', 'Device', 'Card_Type']
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# 3. Standardize Continuous Columns
df['Amount'] = StandardScaler().fit_transform(df[['Amount']])

# 4. Sort by Customer and Time to build chronological history
df['Time'] = pd.to_datetime(df['Time'])
df = df.sort_values(by=['Customer_ID', 'Time']).reset_index(drop=True)

# Define our features and target label
feature_cols = ['Amount', 'Merchant', 'Location', 'Device', 'Previous_Fraud', 'Card_Type']
target_col = 'Label'

In [125]:
df.head()

,Transaction_ID,Customer_ID,Amount,Merchant,Time,Location,Device,Previous_Fraud,Card_Type,Label
0,TXN0043899,CUST00001,0.271323,267,2025-01-01 11:24:11,3,3,0,2,0
1,TXN0034225,CUST00001,-0.015017,249,2025-02-23 20:08:50,29,2,1,0,0
2,TXN0093630,CUST00001,-0.289544,422,2025-03-05 21:09:10,48,3,0,3,0
3,TXN0061819,CUST00001,0.278708,490,2025-04-06 06:09:52,32,0,0,2,0
4,TXN0022888,CUST00001,-0.318624,193,2025-04-08 18:12:50,14,4,0,1,0


In [126]:
def create_sequences(dataframe, feature_cols, target_col, seq_length=5):
    sequences = []
    labels = []
    
    # Group by customer so sequences don't bleed across different users
    for _, group in dataframe.groupby('Customer_ID'):
        features = group[feature_cols].values
        targets = group[target_col].values
        
        for i in range(len(group)):
            # If historical data is shorter than seq_length, pad with zeros
            if i < seq_length - 1:
                pad_len = (seq_length - 1) - i
                padding = np.zeros((pad_len, len(feature_cols)))
                seq = np.vstack([padding, features[:i+1]])
            else:
                seq = features[i - seq_length + 1 : i + 1]
                
            sequences.append(seq)
            labels.append(targets[i])
            
    return np.array(sequences), np.array(labels)

# Generate sequences (e.g., looking at a window of 5 transactions)
SEQ_LENGTH = 5
X_seq, y_seq = create_sequences(df, feature_cols, target_col, seq_length=SEQ_LENGTH)

print("X shape:", X_seq.shape) # Expected: (num_samples, seq_length, num_features)
print("y shape:", y_seq.shape) # Expected: (num_samples,)

X shape: (100000, 5, 6)
y shape: (100000,)


In [127]:
from torch.utils.data import Dataset, DataLoader

class FraudDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # Shape: (batch_size, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Instantiate DataLoader
dataset = FraudDataset(X_seq, y_seq)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [128]:
import torch.nn as nn

class FraudLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(FraudLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        # Note: nn.Sigmoid() is removed from here

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :] 
        
        # Return raw outputs (logits) directly
        return self.fc(out)

# Model parameters
INPUT_SIZE = len(feature_cols) # 6 features
HIDDEN_SIZE = 32
NUM_LAYERS = 2

model = FraudLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS)

In [129]:
import torch.optim as optim

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# 97,000 legitimate transactions / 3,000 fraud transactions = ~32.33
pos_weight = torch.tensor([32.33]).to(device)

# Use BCEWithLogitsLoss and pass the class weights
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 10
# (Your training loop code below this remains exactly the same!)

# Training Loop
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {epoch_loss:.4f}")

Epoch [1/10], Loss: 0.7079
Epoch [2/10], Loss: 0.4885
Epoch [3/10], Loss: 0.4788
Epoch [4/10], Loss: 0.4645
Epoch [5/10], Loss: 0.4637
Epoch [6/10], Loss: 0.4677
Epoch [7/10], Loss: 0.4612
Epoch [8/10], Loss: 0.4575
Epoch [9/10], Loss: 0.4542
Epoch [10/10], Loss: 0.4539


In [130]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_model(model, data_loader, device, threshold=0.30): # Set your custom threshold here
    model.eval()  
    all_preds = []
    all_labels = []
    
    with torch.no_grad():  
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            
            # 1. Forward pass outputs raw logits
            logits = model(inputs)
            
            # 2. Convert logits to probabilities using Sigmoid
            probabilities = torch.sigmoid(logits)
            
            # 3. Apply your custom decision threshold
            preds = (probabilities >= threshold).float()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            
    # Calculate Metrics
    accuracy = accuracy_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)
    
    print(f"Overall Accuracy (Threshold {threshold}): {accuracy * 100:.2f}%\n")
    print("--- Confusion Matrix ---")
    print(cm)
    print("\n--- Detailed Classification Report ---")
    print(classification_report(all_labels, all_preds, target_names=['Legit', 'Fraud']))

In [131]:
# Call the evaluation function
evaluate_model(model, train_loader, device, threshold=0.30)

Overall Accuracy (Threshold 0.3): 81.76%

--- Confusion Matrix ---
[[78929 18071]
 [  167  2833]]

--- Detailed Classification Report ---
              precision    recall  f1-score   support

       Legit       1.00      0.81      0.90     97000
       Fraud       0.14      0.94      0.24      3000

    accuracy                           0.82    100000
   macro avg       0.57      0.88      0.57    100000
weighted avg       0.97      0.82      0.88    100000



In [132]:
def predict_single_input(model, sample_seq, device, threshold=0.30):
    model.eval()
    with torch.no_grad():
        tensor_input = torch.tensor(sample_seq, dtype=torch.float32)
        tensor_input = tensor_input.unsqueeze(0).to(device)
        
        # 1. Get raw logit output
        logit_output = model(tensor_input)
        
        # 2. Convert to an actual probability percentage using Sigmoid
        probability = torch.sigmoid(logit_output).item()
        
        print(f"Analyzed Sequence. Fraud Probability: {probability * 100:.2f}%")
        
        # 3. Evaluate against custom threshold
        if probability >= threshold:
            print(f"❌ ALERT: Transaction flagged as FRAUDULENT (Exceeded {threshold*100}% threshold).")
        else:
            print("✅ Clear: Transaction approved.")

In [133]:
# Call the evaluation function
evaluate_model(model, train_loader, device, threshold=0.30)

Overall Accuracy (Threshold 0.3): 81.76%

--- Confusion Matrix ---
[[78929 18071]
 [  167  2833]]

--- Detailed Classification Report ---
              precision    recall  f1-score   support

       Legit       1.00      0.81      0.90     97000
       Fraud       0.14      0.94      0.24      3000

    accuracy                           0.82    100000
   macro avg       0.57      0.88      0.57    100000
weighted avg       0.97      0.82      0.88    100000



In [136]:
import numpy as np

single_customer_history = np.array([
    [-0.52, 2, 12, 1, 0, 3],  # 4th oldest transaction
    [-0.21, 2, 12, 1, 0, 3],  # 3rd oldest transaction
    [ 0.12, 5, 3,  1, 0, 3],  # 2nd oldest transaction
    [ 1.55, 9, 45, 0, 0, 1],  # Previous transaction
    [ 4.89, 1, 8,  0, 0, 2]   # The CURRENT transaction we want to check
], dtype=np.float32)

In [137]:
# Call the single prediction function 
# (Make sure 'single_customer_history' is defined in a cell above this!)
predict_single_input(model, single_customer_history, device, threshold=0.30)

Analyzed Sequence. Fraud Probability: 99.91%
❌ ALERT: Transaction flagged as FRAUDULENT (Exceeded 30.0% threshold).


In [139]:
# Try a strict threshold of 0.50
evaluate_model(model, train_loader, device, threshold=0.50)

# Try an even higher threshold of 0.75
evaluate_model(model, train_loader, device, threshold=0.75)

Overall Accuracy (Threshold 0.5): 91.24%

--- Confusion Matrix ---
[[88537  8463]
 [  295  2705]]

--- Detailed Classification Report ---
              precision    recall  f1-score   support

       Legit       1.00      0.91      0.95     97000
       Fraud       0.24      0.90      0.38      3000

    accuracy                           0.91    100000
   macro avg       0.62      0.91      0.67    100000
weighted avg       0.97      0.91      0.94    100000

Overall Accuracy (Threshold 0.75): 96.39%

--- Confusion Matrix ---
[[93914  3086]
 [  523  2477]]

--- Detailed Classification Report ---
              precision    recall  f1-score   support

       Legit       0.99      0.97      0.98     97000
       Fraud       0.45      0.83      0.58      3000

    accuracy                           0.96    100000
   macro avg       0.72      0.90      0.78    100000
weighted avg       0.98      0.96      0.97    100000



In [140]:
# Try a very high threshold of 0.85
evaluate_model(model, train_loader, device, threshold=0.85)

Overall Accuracy (Threshold 0.85): 97.79%

--- Confusion Matrix ---
[[95512  1488]
 [  724  2276]]

--- Detailed Classification Report ---
              precision    recall  f1-score   support

       Legit       0.99      0.98      0.99     97000
       Fraud       0.60      0.76      0.67      3000

    accuracy                           0.98    100000
   macro avg       0.80      0.87      0.83    100000
weighted avg       0.98      0.98      0.98    100000



In [141]:
# Create a checkpoint dictionary containing both the weights and your 0.75 threshold
checkpoint = {
    'model_state_dict': model.state_dict(),
    'threshold': 0.75
}

# Save the entire bundle to a file
torch.save(checkpoint, 'lstm_model.pth')
print("Saved model weights and locked-in the 0.75 threshold!")

Saved model weights and locked-in the 0.75 threshold!
